# MARLISA coordinated multi-agent baseline

This notebook trains CityLearn's **MARLISA** controller and compares its deterministic final episode with `BasicRBC` and independent SAC.

The experiment keeps the common setup fixed: the same three buildings, 720-step local dataset, random seed, reward, SAC hyperparameters, exploration schedule, and KPI calculation. The main experimental change is MARLISA's demand estimation and information sharing between building policies.


## SAC versus MARLISA

Both controllers maintain one actor and two critics for every building. Independent SAC chooses each building's action from its local encoded observations. MARLISA adds:

1. a linear model that predicts each building's electricity demand after an action;
2. two coordination variables describing expected district demand and previously dispatched building capacity;
3. iterative action selection that passes coordination information through the building policies; and
4. PCA preprocessing after the initial exploration period.

The default run uses the same `IndependentSACReward` as the SAC notebook. This isolates the effect of coordination more clearly. Selecting CityLearn's `MARL` reward would define a different learning objective and should be reported as a separate experiment.

The installed CityLearn 3.0.2 implementation selects a policy using the coordination-iteration index during post-exploration action selection. With three buildings and two iterations, that bypasses a building policy. The project runner applies a narrow compatibility correction that selects `policy_net[building]`; all other MARLISA calculations remain unchanged. The correction is recorded in the run metadata.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.run_marlisa import run_marlisa

SCHEMA = PROJECT_ROOT / 'notebooks' / 'data' / 'schema.json'
OUTPUT_DIRECTORY = PROJECT_ROOT / 'outputs' / 'marlisa'
print(f'Project root: {PROJECT_ROOT}')
print(f'Schema exists: {SCHEMA.is_file()}')


## Comparable configuration

The SAC settings below match `02_Independent_SAC.ipynb`. `TRAINING_EPISODES = 3` gives one initial exploration episode, one training episode, and one deterministic evaluation episode. It verifies the implementation but is too short for a final result.


In [ ]:
SEED = 42
TRAINING_EPISODES = 3
REWARD = 'independent_sac'

SAC_PARAMETERS = {
    'hidden_dimension': (256, 256),
    'discount': 0.99,
    'tau': 0.005,
    'alpha': 0.2,
    'learning_rate': 3e-4,
    'batch_size': 256,
    'replay_buffer_capacity': 100_000,
    'action_scaling_coefficient': 0.5,
    'reward_scaling': 5.0,
    'update_per_time_step': 1,
}

MARLISA_PARAMETERS = {
    'regression_buffer_capacity': 30_000,
    'start_regression_time_step': 2,
    'regression_frequency': 2_500,
    'information_sharing': True,
    'pca_compression': 1.0,
    'iterations': 2,
}

pd.Series({
    'seed': SEED,
    'episodes': TRAINING_EPISODES,
    'reward': REWARD,
    **SAC_PARAMETERS,
    **MARLISA_PARAMETERS,
})


## Train and evaluate MARLISA

This is slower than independent SAC because MARLISA also fits demand models, applies PCA, and performs iterative coordinated action selection. The final episode uses deterministic actor outputs and does not update the networks.


In [ ]:
result = run_marlisa(
    schema=SCHEMA,
    output_directory=OUTPUT_DIRECTORY,
    episodes=TRAINING_EPISODES,
    random_seed=SEED,
    reward=REWARD,
    **SAC_PARAMETERS,
    **MARLISA_PARAMETERS,
)

env = result['env']
agent = result['agent']
print(f'Device: {agent.device}')
print(f'Building policies: {len(agent.policy_net)}')
print(f'Reward function: {type(env.reward_function).__name__}')
print(f'Information sharing: {agent.information_sharing}')
print(f'Demand estimators fitted: {agent.regression_flag}')
print(f'PCA fitted: {agent.pca_flag}')
print(f"Outputs: {result['output_directory']}")


## Episode rewards

Reward is useful for monitoring learning within one fixed reward definition. The final point is deterministic evaluation. Do not compare reward numbers from runs that use different reward functions.


In [ ]:
training_rewards = result['training_rewards']
display(training_rewards.round(3))

reward_plot = training_rewards.pivot(index='episode', columns='building', values='sum')
ax = reward_plot.plot(marker='o', figsize=(9, 4))
ax.set_title('MARLISA episode reward by building')
ax.set_ylabel('Sum of rewards (higher is better)')
ax.grid(alpha=0.25)
plt.show()


## MARLISA KPI results

The KPI table describes the final deterministic episode. CityLearn reports electricity, carbon, cost, peaks, ramping, and thermal discomfort independently of the reward used for learning. For normalized electricity KPIs, values below 1 improve on the corresponding no-control reference.


In [ ]:
display(result['kpi_table'].round(3))
display(result['fairness'].round(3))


## Compare RBC, independent SAC, and MARLISA

The runner automatically loads existing RBC and SAC KPI files. Run notebooks `01` and `02` first if a controller column is absent. RBC has no learning objective, so its behavior is unaffected by the environment reward. SAC and MARLISA use the same reward in this comparison.


In [ ]:
metadata_sources = {
    'BasicRBC': PROJECT_ROOT / 'outputs' / 'rbc' / 'metadata.json',
    'Independent SAC': PROJECT_ROOT / 'outputs' / 'independent_sac' / 'metadata.json',
}
contexts = []
for controller, path in metadata_sources.items():
    if path.is_file():
        metadata = json.loads(path.read_text(encoding='utf-8'))
        contexts.append({
            'controller': controller,
            'seed': metadata.get('random_seed'),
            'time_steps': metadata.get('episode_time_steps', metadata.get('time_steps')),
            'episodes': metadata.get('episodes', 1),
            'reward': metadata.get('reward_choice', 'not used for RBC actions'),
        })
contexts.append({
    'controller': 'MARLISA',
    'seed': result['metadata']['random_seed'],
    'time_steps': result['metadata']['episode_time_steps'],
    'episodes': result['metadata']['episodes'],
    'reward': result['metadata']['reward_choice'],
})
display(pd.DataFrame(contexts).set_index('controller'))

KEY_DISTRICT_KPIS = [
    'carbon_emissions_total',
    'cost_total',
    'electricity_consumption_total',
    'daily_peak_average',
    'ramping_average',
]

comparison = result['comparison']
available_kpis = [name for name in KEY_DISTRICT_KPIS if name in comparison.index]
display(comparison.loc[available_kpis].round(3))

ax = comparison.loc[available_kpis].plot.bar(figsize=(11, 5))
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='No-control reference')
ax.set_ylabel('KPI value (lower is better)')
ax.set_title('District KPI comparison')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## Compare building-level discomfort fairness

The project defines operational fairness using the worst building's discomfort and the gap between the most and least uncomfortable buildings. Smaller values are better. This is fairness between simulated buildings, not demographic fairness.


In [ ]:
controller_files = {
    'BasicRBC': PROJECT_ROOT / 'outputs' / 'rbc' / 'kpis.csv',
    'Independent SAC': PROJECT_ROOT / 'outputs' / 'independent_sac' / 'kpis.csv',
    'MARLISA': PROJECT_ROOT / 'outputs' / 'marlisa' / 'kpis.csv',
}

fairness_rows = []
for controller, path in controller_files.items():
    if not path.is_file():
        continue
    kpis = pd.read_csv(path)
    discomfort = kpis.loc[
        (kpis['level'] == 'building')
        & (kpis['cost_function'] == 'discomfort_proportion'),
        ['name', 'value'],
    ].dropna()
    if discomfort.empty:
        continue
    worst = discomfort.loc[discomfort['value'].idxmax()]
    fairness_rows.append({
        'controller': controller,
        'average_discomfort': discomfort['value'].mean(),
        'worst_discomfort': discomfort['value'].max(),
        'discomfort_gap': discomfort['value'].max() - discomfort['value'].min(),
        'worst_building': worst['name'],
    })

fairness_comparison = pd.DataFrame(fairness_rows).set_index('controller')
display(fairness_comparison.round(3))


## Inspect coordinated actions

The action table contains the final deterministic episode. Look for differences between buildings and time periods. Coordination does not require identical actions; MARLISA uses district context to let each building choose an action suited to its own state.


In [ ]:
actions = result['actions']
display(actions.head(12))

first_day = actions.query('time_step < 24')
for building_name, building_actions in first_day.groupby('building'):
    action_plot = building_actions.pivot(index='time_step', columns='action', values='value')
    ax = action_plot.plot(figsize=(10, 3), title=f'{building_name}: first 24 MARLISA control steps')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Action')
    ax.grid(alpha=0.25)
    plt.show()


## Interpretation and next experiment

A three-episode result only confirms that the pipeline operates. For the report:

1. define separate training and held-out evaluation periods;
2. train SAC and MARLISA for the same number of transitions;
3. run both learned controllers with the same set of random seeds;
4. report mean and variation for carbon, cost, peak demand, average discomfort, worst-building discomfort, and discomfort gap; and
5. change one design choice at a time when introducing the carbon-aware and fairness rewards.

Generated files are stored in `outputs/marlisa/`, including training rewards, final actions and rewards, KPIs, fairness summary, controller comparison, metadata, and a checkpoint.
